# Entscheidungen unter Unsicherheit: Personalplanung im Krankenhaus

**Begleitmaterial zur Vorlesungseinheit** — Dr. Celso Gustavo Stall Sikora

Dieses Notebook enthält das vollständige Rechenbeispiel aus der Vorlesung:
Eine Station plant die Nachtdienste für die Samstagnächte des kommenden Monats.
Festgelegt werden **jetzt**: $x_V$ Vollarbeitskräfte und $x_B$ Bereitschaftsdienste.
Der Bedarf $D$ zeigt sich **erst in der Nacht**; reagiert wird durch Aktivierung
der Bereitschaft und — als teuerste Stufe — durch **Einspringen**.

$$\min_{x_V,\; x_B \le 2}\; \underbrace{1{,}0\,x_V + 0{,}75\,x_B}_{\text{jetzt, sicher}}
\;+\; \underbrace{\mathbb{E}_D\big[\,0{,}3 \cdot B(D) + 3{,}0 \cdot S(D)\,\big]}_{\text{später, im Erwartungswert}}$$

mit $B(D) = \min(\max(D-x_V,0),\, x_B)$ (aktivierte Bereitschaft) und
$S(D) = \max(D - x_V - x_B,\, 0)$ (Einspringer).

**Kostenstruktur** (stilisiert nach TV-Ärzte/VKA, alle Werte in "Schichtäquivalenten"):

| Instrument | Kosten |
|---|---|
| Vollarbeit | 1,0 pro Schicht |
| Bereitschaftsdienst (Stufe II, § 12) | 0,75 + 0,3 je Einsatz (Stufendrift; aktiviert 1,05 > 1,0) |
| Einspringen / kurzfristiger Ersatz | 3,0 |

**Ihre Aufgabe zu Hause:** Ändern Sie die Parameter im nächsten Block und beobachten
Sie, wie sich der optimale Dienstplan verschiebt. Die Leitfragen stehen am Ende.

In [ ]:
# ============================================================
# PARAMETER — hier dürfen (sollen!) Sie ändern und neu rechnen
# ============================================================

# Kosten (in Schichtäquivalenten)
c_V   = 1.0    # Vollarbeit pro Schicht
c_B   = 0.75   # Bereitschaftsdienst: Grundkosten (Stufe II = 75 %)
c_akt = 0.3    # Zusatzkosten je aktivierter Bereitschaft (Stufendrift; aktiviert: 0,75+0,3 > 1,0)
c_E   = 3.0    # Einspringen / kurzfristiger Ersatz

# Obergrenze Bereitschaftsdienste (§ 10: keine Regelarbeit)
x_B_max = 2

# Bedarfsverteilung: D -> Wahrscheinlichkeit
verteilung = {2: 0.05, 3: 0.20, 4: 0.30, 5: 0.25, 6: 0.15, 7: 0.05}

assert abs(sum(verteilung.values()) - 1.0) < 1e-9, "Wahrscheinlichkeiten müssen sich zu 1 summieren!"
E_D = sum(d * p for d, p in verteilung.items())
print(f"Erwarteter Bedarf E[D] = {E_D:.2f} Pflegekräfte")

In [ ]:
# ============================================================
# MODELL: erwartete Gesamtkosten eines Plans (x_V, x_B)
# ============================================================

def kosten(x_V, x_B, detail=False):
    # Erwartete Gesamtkosten des Plans (x_V Vollarbeit, x_B Bereitschaft)
    plankosten = c_V * x_V + c_B * x_B
    e_akt, e_einspr = 0.0, 0.0
    for d, p in verteilung.items():
        fehl = max(d - x_V, 0)                 # Unterdeckung nach Vollarbeit
        aktiviert = min(fehl, x_B)             # Bereitschaft einspringen lassen
        einspringer = max(d - x_V - x_B, 0)    # Rest: Einspringen
        e_akt += p * aktiviert
        e_einspr += p * einspringer
    gesamt = plankosten + c_akt * e_akt + c_E * e_einspr
    if detail:
        return dict(plan=plankosten, aktivierung=c_akt * e_akt,
                    einspringen=c_E * e_einspr, gesamt=gesamt,
                    p_einspringen=sum(p for d, p in verteilung.items() if d > x_V + x_B))
    return gesamt

# Beispiel: der Plan aus der Vorlesung
print(kosten(3, 2, detail=True))

In [ ]:
# ============================================================
# VOLLSTÄNDIGE ENUMERATION: alle Pläne im Gitter
# ============================================================
import numpy as np

D_max = max(verteilung)
gitter = {(v, b): kosten(v, b)
          for v in range(0, D_max + 1)
          for b in range(0, x_B_max + 1)}

optimum = min(gitter, key=gitter.get)
print(f"Optimaler Plan: {optimum[0]} Vollarbeit + {optimum[1]} Bereitschaft "
      f"-> erwartete Kosten {gitter[optimum]:.2f}")

print("\nKostenmatrix (Zeilen: x_V, Spalten: x_B):")
print("x_V\\x_B " + "".join(f"{b:>8}" for b in range(x_B_max + 1)))
for v in range(0, D_max + 1):
    zeile = "".join(f"{gitter[(v, b)]:8.2f}" for b in range(x_B_max + 1))
    marker = "  <-- Optimum" if v == optimum[0] else ""
    print(f"{v:>4}   {zeile}{marker}")

In [ ]:
# ============================================================
# VERGLEICH: die drei Pläne aus der Vorlesung + Optimum
# ============================================================
import matplotlib.pyplot as plt

plaene = {
    f"Plan A 'Durchschnitt' ({round(E_D)}V)": (round(E_D), 0),
    f"Plan B 'Auf Nummer sicher' ({D_max - 1}V)": (D_max - 1, 0),
    f"Plan C 'Portfolio' ({optimum[0]}V+{optimum[1]}B)": optimum,
}

fig, ax = plt.subplots(figsize=(9, 3.2))
farben = {"plan": "#00547a", "aktivierung": "#e0a96d", "einspringen": "#cc5500"}
for i, (name, (v, b)) in enumerate(plaene.items()):
    det = kosten(v, b, detail=True)
    links = 0
    for teil, wert in [("plan", det["plan"]), ("aktivierung", det["aktivierung"]),
                       ("einspringen", det["einspringen"])]:
        ax.barh(i, wert, left=links, color=farben[teil], edgecolor="white")
        links += wert
    ax.text(links + 0.05, i, f"{det['gesamt']:.2f}", va="center", fontweight="bold")
ax.set_yticks(range(len(plaene)), plaene.keys())
ax.invert_yaxis()
ax.set_xlabel("Erwartete Kosten pro Nacht (Schichtäquivalente)")
ax.legend(["Plankosten", "Bereitschaft aktiviert", "Einspringen"], loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

for name, (v, b) in plaene.items():
    det = kosten(v, b, detail=True)
    print(f"{name}: gesamt {det['gesamt']:.2f} | P(Einspringen) = {det['p_einspringen']:.0%}")

In [ ]:
# ============================================================
# VALUE OF THE STOCHASTIC SOLUTION (VSS)
# ============================================================
import math

# Deterministische Logik: plane für E[D] (aufgerundet), nur Vollarbeit —
# unter Sicherheit gäbe es rechtlich gar keinen Bereitschaftsdienst.
x_det = math.ceil(E_D)
eev = kosten(x_det, 0)
vss = eev - gitter[optimum]
print(f"Deterministischer Plan ({x_det} Vollarbeit): {eev:.2f}")
print(f"Stochastisches Optimum {optimum}:          {gitter[optimum]:.2f}")
print(f"Value of the Stochastic Solution (VSS):    {vss:.2f}  ({vss/eev:.1%})")

## Aufgaben

1. **Einspringpreis.** Senken Sie $c_E$ schrittweise von 3,0 auf 1,0.
   Ab welchem Wert verschwindet die Bereitschaft aus dem optimalen Plan? Warum?
2. **Stufendrift.** Setzen Sie $c_{akt} = 0$. Was passiert mit dem Optimum —
   und warum ist das Ergebnis ökonomisch (und tarifrechtlich) unplausibel?
3. **Verteilung.** Ersetzen Sie die Verteilung durch eine symmetrische mit gleichem
   Erwartungswert (z. B. $\{3{:}\,0{,}2;\ 4{:}\,0{,}3;\ 5{:}\,0{,}3;\ 6{:}\,0{,}2\}$).
   Wie ändert sich der Wert des Portfolios? Was lernt man daraus über die Rolle des
   *rechten Rands* der Verteilung?
4. **Kritisches Verhältnis.** Für $x_B = 0$ ist das Problem ein klassisches
   Newsvendor-Problem. Leiten Sie das optimale $x_V$ über das kritische Verhältnis
   $c_E / (c_E + c_V)$ her und vergleichen Sie mit der Enumeration.

## Weiterführende Literatur

- Birge & Louveaux: *Introduction to Stochastic Programming*, Springer (Kap. 1–4).
- Erhard, Schoenfelder, Fügener, Brunner (2018): *State of the art in physician
  scheduling*, EJOR 265(1), 1–18.
- Ban & Rudin (2019): *The Big Data Newsvendor*, Management Science 65(9).
- Porrmann & Römer (2021): *Learning to Reduce State-Expanded Networks for
  Multi-Activity Shift Scheduling*, CPAIOR 2021 — ML-gestütztes Lösen von
  Dienstplanungsmodellen, entwickelt an der Universität Bielefeld.

*Kostenwerte stilisiert nach TV-Ärzte/VKA (§§ 10–12); Einspringprämien nach
öffentlich dokumentierten Regelungen (z. B. LMU Klinikum München).*